<a href="https://colab.research.google.com/github/dorhoffman/SWINGPULSE/blob/main/notebooks/07_Live_Yahoo_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import yfinance as yf

print(yf.__version__)

0.2.66


In [16]:
import os

print(os.getcwd())
print(os.listdir("/content"))

/content
['.config', 'drive', 'sample_data']


In [18]:
from google.colab import drive

drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [19]:
drive.mount("/content/drive", force_remount=True)

ValueError: Mountpoint must not already contain files

In [17]:
print(os.path.exists("/content/drive"))
print(os.path.exists("/content/drive/MyDrive"))

True
True


In [4]:
import pandas as pd

print(pd.__version__)

3.0.3


In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

def download_live_stock(symbol):
    data = yf.download(
        symbol,
        period="2y",
        interval="1d",
        auto_adjust=False,
        actions=True,
        progress=False,
        threads=False
    )

    if data.empty:
        raise ValueError(f"No data returned for {symbol}")

    data = data.reset_index()

    print("Shape:", data.shape)
    print("Columns:", data.columns.tolist())

    return data

In [2]:
aapl_live = download_live_stock("AAPL")
display(aapl_live.tail())

Shape: (501, 9)
Columns: [('Date', ''), ('Adj Close', 'AAPL'), ('Close', 'AAPL'), ('Dividends', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Stock Splits', 'AAPL'), ('Volume', 'AAPL')]


Price,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
496,2026-07-06,312.660004,312.660004,0.0,314.200012,307.000000,307.359985,0.0,53590000
497,2026-07-07,310.660004,310.660004,0.0,315.480011,310.149994,315.290009,0.0,42490000
498,2026-07-08,313.390015,313.390015,0.0,314.820007,307.049988,311.910004,0.0,41323500
499,2026-07-09,316.220001,316.220001,0.0,316.529999,308.160004,310.510010,0.0,48124500
500,2026-07-10,315.320007,315.320007,0.0,316.910004,312.170013,314.720001,0.0,34109200


In [4]:
# Flatten MultiIndex columns
if isinstance(aapl_live.columns, pd.MultiIndex):
    aapl_live.columns = aapl_live.columns.get_level_values(0)

# Remove Adj Close
if "Adj Close" in aapl_live.columns:
    aapl_live = aapl_live.drop(columns=["Adj Close"])

# Clean column names
aapl_live.columns = [
    str(col).strip().replace(" ", "_")
    for col in aapl_live.columns
]

print(aapl_live.columns.tolist())

display(aapl_live.tail())

['Date', 'Close', 'Dividends', 'High', 'Low', 'Open', 'Stock_Splits', 'Volume']


,Date,Close,Dividends,High,Low,Open,Stock_Splits,Volume
496,2026-07-06,312.660004,0.0,314.200012,307.000000,307.359985,0.0,53590000
497,2026-07-07,310.660004,0.0,315.480011,310.149994,315.290009,0.0,42490000
498,2026-07-08,313.390015,0.0,314.820007,307.049988,311.910004,0.0,41323500
499,2026-07-09,316.220001,0.0,316.529999,308.160004,310.510010,0.0,48124500
500,2026-07-10,315.320007,0.0,316.910004,312.170013,314.720001,0.0,34109200


In [5]:
data = download_live_stock("AAPL")

Shape: (501, 9)
Columns: [('Date', ''), ('Adj Close', 'AAPL'), ('Close', 'AAPL'), ('Dividends', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Stock Splits', 'AAPL'), ('Volume', 'AAPL')]


In [6]:
import os

PROJECT_PATH = "/content/drive/MyDrive/SWINGPULSE"
UTILS_FOLDER = f"{PROJECT_PATH}/utils"

os.makedirs(UTILS_FOLDER, exist_ok=True)

FEATURE_MODULE_PATH = (
    f"{UTILS_FOLDER}/feature_engineering.py"
)

print(FEATURE_MODULE_PATH)

/content/drive/MyDrive/SWINGPULSE/utils/feature_engineering.py


In [7]:
%%writefile /content/drive/MyDrive/SWINGPULSE/utils/feature_engineering.py

import numpy as np
import pandas as pd


def calculate_rsi(
    series: pd.Series,
    period: int = 14
) -> pd.Series:
    """Calculate RSI using Wilder-style exponential averages."""

    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()

    avg_loss = loss.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()

    rs = avg_gain / avg_loss

    return 100 - (100 / (1 + rs))


def add_technical_features(
    stock_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Add the same technical features used during model training.

    Required columns:
    Date, Open, High, Low, Close, Volume
    """

    required_columns = {
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    }

    missing_columns = (
        required_columns - set(stock_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    stock_df = stock_df.copy()

    stock_df["Date"] = pd.to_datetime(
        stock_df["Date"],
        errors="coerce"
    )

    numeric_columns = [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    for column in numeric_columns:
        stock_df[column] = pd.to_numeric(
            stock_df[column],
            errors="coerce"
        )

    stock_df = (
        stock_df
        .dropna(
            subset=[
                "Date",
                "Open",
                "High",
                "Low",
                "Close",
                "Volume"
            ]
        )
        .sort_values("Date")
        .drop_duplicates(subset=["Date"])
        .reset_index(drop=True)
    )

    # Returns and volume change
    stock_df["Daily_Return"] = (
        stock_df["Close"].pct_change()
    )

    stock_df["Volume_Change"] = (
        stock_df["Volume"].pct_change()
    )

    # Simple moving averages
    stock_df["SMA_20"] = (
        stock_df["Close"]
        .rolling(window=20)
        .mean()
    )

    stock_df["SMA_50"] = (
        stock_df["Close"]
        .rolling(window=50)
        .mean()
    )

    stock_df["SMA_200"] = (
        stock_df["Close"]
        .rolling(window=200)
        .mean()
    )

    # Exponential moving averages
    stock_df["EMA_20"] = (
        stock_df["Close"]
        .ewm(span=20, adjust=False)
        .mean()
    )

    stock_df["EMA_50"] = (
        stock_df["Close"]
        .ewm(span=50, adjust=False)
        .mean()
    )

    stock_df["EMA_200"] = (
        stock_df["Close"]
        .ewm(span=200, adjust=False)
        .mean()
    )

    # RSI
    stock_df["RSI_14"] = calculate_rsi(
        stock_df["Close"],
        period=14
    )

    # MACD
    ema_12 = (
        stock_df["Close"]
        .ewm(span=12, adjust=False)
        .mean()
    )

    ema_26 = (
        stock_df["Close"]
        .ewm(span=26, adjust=False)
        .mean()
    )

    stock_df["MACD"] = ema_12 - ema_26

    stock_df["MACD_Signal"] = (
        stock_df["MACD"]
        .ewm(span=9, adjust=False)
        .mean()
    )

    stock_df["MACD_Histogram"] = (
        stock_df["MACD"]
        - stock_df["MACD_Signal"]
    )

    # Bollinger Bands
    rolling_mean = (
        stock_df["Close"]
        .rolling(window=20)
        .mean()
    )

    rolling_std = (
        stock_df["Close"]
        .rolling(window=20)
        .std()
    )

    stock_df["BB_Middle"] = rolling_mean
    stock_df["BB_Upper"] = (
        rolling_mean + 2 * rolling_std
    )
    stock_df["BB_Lower"] = (
        rolling_mean - 2 * rolling_std
    )

    stock_df["BB_Width"] = (
        stock_df["BB_Upper"]
        - stock_df["BB_Lower"]
    ) / stock_df["BB_Middle"]

    # ATR
    previous_close = stock_df["Close"].shift(1)

    true_range = pd.concat(
        [
            stock_df["High"] - stock_df["Low"],
            (
                stock_df["High"]
                - previous_close
            ).abs(),
            (
                stock_df["Low"]
                - previous_close
            ).abs()
        ],
        axis=1
    ).max(axis=1)

    stock_df["ATR_14"] = (
        true_range
        .rolling(window=14)
        .mean()
    )

    # Volatility
    stock_df["Volatility_20"] = (
        stock_df["Daily_Return"]
        .rolling(window=20)
        .std()
    )

    # Relative features
    stock_df["Price_to_SMA20"] = (
        stock_df["Close"]
        / stock_df["SMA_20"]
    )

    stock_df["Price_to_SMA50"] = (
        stock_df["Close"]
        / stock_df["SMA_50"]
    )

    stock_df["Price_to_EMA20"] = (
        stock_df["Close"]
        / stock_df["EMA_20"]
    )

    stock_df["EMA20_to_EMA50"] = (
        stock_df["EMA_20"]
        / stock_df["EMA_50"]
    )

    stock_df = stock_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return stock_df

Writing /content/drive/MyDrive/SWINGPULSE/utils/feature_engineering.py


In [8]:
import sys
import importlib

UTILS_FOLDER = (
    "/content/drive/MyDrive/SWINGPULSE/utils"
)

if UTILS_FOLDER not in sys.path:
    sys.path.append(UTILS_FOLDER)

import feature_engineering
importlib.reload(feature_engineering)

from feature_engineering import (
    add_technical_features
)

print("Feature module imported successfully")

Feature module imported successfully


In [9]:
import yfinance as yf
import pandas as pd


def download_live_stock(
    symbol: str,
    period: str = "2y"
) -> pd.DataFrame:
    symbol = str(symbol).upper().strip()

    data = yf.download(
        symbol,
        period=period,
        interval="1d",
        auto_adjust=False,
        actions=True,
        progress=False,
        threads=False
    )

    if data.empty:
        raise ValueError(
            f"No Yahoo Finance data returned "
            f"for {symbol}."
        )

    data = data.reset_index()

    # yfinance may return MultiIndex columns
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = (
            data.columns
            .get_level_values(0)
        )

    data.columns = [
        str(column)
        .strip()
        .replace(" ", "_")
        for column in data.columns
    ]

    # The training pipeline used Close, not Adj_Close
    if "Adj_Close" in data.columns:
        data = data.drop(
            columns=["Adj_Close"]
        )

    data["Symbol"] = symbol

    required_columns = [
        "Date",
        "Symbol",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Yahoo data is missing columns: "
            f"{missing_columns}"
        )

    return data

In [10]:
aapl_live = download_live_stock("AAPL")

aapl_live_features = add_technical_features(
    aapl_live
)

print("Downloaded rows:", len(aapl_live))
print(
    "Latest live date:",
    aapl_live_features["Date"].max()
)

display(
    aapl_live_features[
        [
            "Date",
            "Symbol",
            "Close",
            "SMA_200",
            "RSI_14",
            "MACD",
            "MACD_Signal",
            "ATR_14",
            "Volatility_20"
        ]
    ].tail()
)

Downloaded rows: 501
Latest live date: 2026-07-10 00:00:00


,Date,Symbol,Close,SMA_200,RSI_14,MACD,MACD_Signal,ATR_14,Volatility_20
496,2026-07-06,AAPL,312.660004,271.063999,62.430222,0.878922,-0.581901,8.918577,0.024386
497,2026-07-07,AAPL,310.660004,271.422349,60.661242,1.935477,-0.078425,8.824293,0.024252
498,2026-07-08,AAPL,313.390015,271.799899,62.234300,2.958984,0.529057,8.914294,0.023856
499,2026-07-09,AAPL,316.220001,272.153499,63.848136,3.952911,1.213827,8.961435,0.022079
500,2026-07-10,AAPL,315.320007,272.449699,62.927178,4.614786,1.894019,8.946433,0.022139


In [12]:
import os

MODELS_FOLDER = "/content/drive/MyDrive/SWINGPULSE/models"

print("Folder exists:", os.path.exists(MODELS_FOLDER))

if os.path.exists(MODELS_FOLDER):
    print("\nFiles:")
    for f in os.listdir(MODELS_FOLDER):
        print(f)

Folder exists: False


In [13]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith(".joblib"):
            print(os.path.join(root, file))

In [14]:
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith(".json"):
            print(os.path.join(root, file))

In [1]:
import os

print(os.listdir("/content"))

['.config', 'drive', 'sample_data']


In [2]:
import os
import shutil

# מחיקת תיקיית mount התקולה
if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive")

print("Broken drive folder removed")

Broken drive folder removed


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import os

print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive")[:10])

MyDrive exists: True
['Invoice_4559_4340.htm', 'Invoice_4559_4340.gdoc', 'Dor Hoffman CV 2024(1).docx.pdf', 'Dor Hoffman CV 2024(1).docx.gdoc', 'DFD update.drawio', 'IMG_4119.png', 'plant_team2.3mf', 'Electronic ticket receipt, September 02 for DOR BARUCH HOFFMAN.pdf', '206553588_2025_08.pdf', 'Shaked Refua CV  U.gdoc']


In [5]:
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith((".joblib", ".json")):
            print(os.path.join(root, file))

/content/drive/MyDrive/SWINGPULSE/models/logistic_regression_model.joblib
/content/drive/MyDrive/SWINGPULSE/models/random_forest_model.joblib
/content/drive/MyDrive/SWINGPULSE/models/model_metadata.json


In [7]:
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith((".joblib", ".json")):
            print(os.path.join(root, file))

/content/drive/MyDrive/SWINGPULSE/models/logistic_regression_model.joblib
/content/drive/MyDrive/SWINGPULSE/models/random_forest_model.joblib
/content/drive/MyDrive/SWINGPULSE/models/model_metadata.json


In [8]:
import json
import joblib

MODELS_FOLDER = "/content/drive/MyDrive/SWINGPULSE/models"

MODEL_PATH = (
    f"{MODELS_FOLDER}/"
    "random_forest_model.joblib"
)

METADATA_PATH = (
    f"{MODELS_FOLDER}/"
    "model_metadata.json"
)

model = joblib.load(MODEL_PATH)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:
    metadata = json.load(file)

FEATURE_COLUMNS = metadata["features"]
DECISION_THRESHOLD = metadata["decision_threshold"]
PREDICTION_HORIZON = metadata["prediction_horizon"]
TARGET_RETURN = metadata["target_return"]

print("Model loaded successfully")
print("Selected model:", metadata["selected_model"])
print("Number of features:", len(FEATURE_COLUMNS))
print("Decision threshold:", DECISION_THRESHOLD)
print("Target:", f"+{TARGET_RETURN * 100:.0f}%")
print("Horizon:", PREDICTION_HORIZON, "trading days")

Model loaded successfully
Selected model: Random Forest
Number of features: 19
Decision threshold: 0.45
Target: +10%
Horizon: 10 trading days


In [9]:
analyze_live_stock()

NameError: name 'analyze_live_stock' is not defined

In [10]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import yfinance as yf

PROJECT_PATH = "/content/drive/MyDrive/SWINGPULSE"

MODELS_FOLDER = f"{PROJECT_PATH}/models"
UTILS_FOLDER = f"{PROJECT_PATH}/utils"

MODEL_PATH = (
    f"{MODELS_FOLDER}/"
    "random_forest_model.joblib"
)

METADATA_PATH = (
    f"{MODELS_FOLDER}/"
    "model_metadata.json"
)

print("Model exists:", os.path.exists(MODEL_PATH))
print("Metadata exists:", os.path.exists(METADATA_PATH))
print("Utils folder exists:", os.path.exists(UTILS_FOLDER))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model exists: True
Metadata exists: True
Utils folder exists: False


In [12]:
import os

PROJECT_PATH = "/content/drive/MyDrive/SWINGPULSE"
UTILS_FOLDER = f"{PROJECT_PATH}/utils"

os.makedirs(UTILS_FOLDER, exist_ok=True)

print("Utils folder exists:", os.path.exists(UTILS_FOLDER))
print("Utils path:", UTILS_FOLDER)

Utils folder exists: True
Utils path: /content/drive/MyDrive/SWINGPULSE/utils


In [13]:
%%writefile /content/drive/MyDrive/SWINGPULSE/utils/feature_engineering.py

import numpy as np
import pandas as pd


def calculate_rsi(series, period=14):
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()

    avg_loss = loss.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()

    rs = avg_gain / avg_loss

    return 100 - (100 / (1 + rs))


def add_technical_features(stock_df):
    required_columns = {
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    }

    missing_columns = (
        required_columns - set(stock_df.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    stock_df = stock_df.copy()

    stock_df["Date"] = pd.to_datetime(
        stock_df["Date"],
        errors="coerce"
    )

    numeric_columns = [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    for column in numeric_columns:
        stock_df[column] = pd.to_numeric(
            stock_df[column],
            errors="coerce"
        )

    stock_df = (
        stock_df
        .dropna(
            subset=[
                "Date",
                "Open",
                "High",
                "Low",
                "Close",
                "Volume"
            ]
        )
        .sort_values("Date")
        .drop_duplicates(subset=["Date"])
        .reset_index(drop=True)
    )

    stock_df["Daily_Return"] = (
        stock_df["Close"].pct_change()
    )

    stock_df["Volume_Change"] = (
        stock_df["Volume"].pct_change()
    )

    stock_df["SMA_20"] = (
        stock_df["Close"]
        .rolling(window=20)
        .mean()
    )

    stock_df["SMA_50"] = (
        stock_df["Close"]
        .rolling(window=50)
        .mean()
    )

    stock_df["SMA_200"] = (
        stock_df["Close"]
        .rolling(window=200)
        .mean()
    )

    stock_df["EMA_20"] = (
        stock_df["Close"]
        .ewm(span=20, adjust=False)
        .mean()
    )

    stock_df["EMA_50"] = (
        stock_df["Close"]
        .ewm(span=50, adjust=False)
        .mean()
    )

    stock_df["EMA_200"] = (
        stock_df["Close"]
        .ewm(span=200, adjust=False)
        .mean()
    )

    stock_df["RSI_14"] = calculate_rsi(
        stock_df["Close"],
        period=14
    )

    ema_12 = (
        stock_df["Close"]
        .ewm(span=12, adjust=False)
        .mean()
    )

    ema_26 = (
        stock_df["Close"]
        .ewm(span=26, adjust=False)
        .mean()
    )

    stock_df["MACD"] = ema_12 - ema_26

    stock_df["MACD_Signal"] = (
        stock_df["MACD"]
        .ewm(span=9, adjust=False)
        .mean()
    )

    stock_df["MACD_Histogram"] = (
        stock_df["MACD"]
        - stock_df["MACD_Signal"]
    )

    rolling_mean = (
        stock_df["Close"]
        .rolling(window=20)
        .mean()
    )

    rolling_std = (
        stock_df["Close"]
        .rolling(window=20)
        .std()
    )

    stock_df["BB_Middle"] = rolling_mean

    stock_df["BB_Upper"] = (
        rolling_mean + 2 * rolling_std
    )

    stock_df["BB_Lower"] = (
        rolling_mean - 2 * rolling_std
    )

    stock_df["BB_Width"] = (
        stock_df["BB_Upper"]
        - stock_df["BB_Lower"]
    ) / stock_df["BB_Middle"]

    previous_close = stock_df["Close"].shift(1)

    true_range = pd.concat(
        [
            stock_df["High"] - stock_df["Low"],
            (
                stock_df["High"]
                - previous_close
            ).abs(),
            (
                stock_df["Low"]
                - previous_close
            ).abs()
        ],
        axis=1
    ).max(axis=1)

    stock_df["ATR_14"] = (
        true_range
        .rolling(window=14)
        .mean()
    )

    stock_df["Volatility_20"] = (
        stock_df["Daily_Return"]
        .rolling(window=20)
        .std()
    )

    stock_df["Price_to_SMA20"] = (
        stock_df["Close"]
        / stock_df["SMA_20"]
    )

    stock_df["Price_to_SMA50"] = (
        stock_df["Close"]
        / stock_df["SMA_50"]
    )

    stock_df["Price_to_EMA20"] = (
        stock_df["Close"]
        / stock_df["EMA_20"]
    )

    stock_df["EMA20_to_EMA50"] = (
        stock_df["EMA_20"]
        / stock_df["EMA_50"]
    )

    stock_df = stock_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return stock_df

Writing /content/drive/MyDrive/SWINGPULSE/utils/feature_engineering.py


In [14]:
import os

FEATURE_ENGINEERING_PATH = (
    "/content/drive/MyDrive/SWINGPULSE/"
    "utils/feature_engineering.py"
)

print(
    "Feature module exists:",
    os.path.exists(FEATURE_ENGINEERING_PATH)
)

print(
    "File size:",
    os.path.getsize(FEATURE_ENGINEERING_PATH),
    "bytes"
)

Feature module exists: True
File size: 4544 bytes


In [15]:
import sys
import importlib

UTILS_FOLDER = (
    "/content/drive/MyDrive/SWINGPULSE/utils"
)

if UTILS_FOLDER not in sys.path:
    sys.path.append(UTILS_FOLDER)

import feature_engineering
importlib.reload(feature_engineering)

from feature_engineering import (
    add_technical_features
)

print("Feature engineering module loaded successfully")

Feature engineering module loaded successfully


In [16]:
print("Utils folder exists:", os.path.exists(UTILS_FOLDER))

Utils folder exists: True


In [17]:
import yfinance as yf
import pandas as pd


def download_live_stock(
    symbol: str,
    period: str = "2y"
) -> pd.DataFrame:
    symbol = str(symbol).upper().strip()

    data = yf.download(
        symbol,
        period=period,
        interval="1d",
        auto_adjust=False,
        actions=True,
        progress=False,
        threads=False
    )

    if data.empty:
        raise ValueError(
            f"No Yahoo Finance data returned for {symbol}."
        )

    data = data.reset_index()

    # טיפול ב־MultiIndex של yfinance
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data.columns = [
        str(column).strip().replace(" ", "_")
        for column in data.columns
    ]

    # המודל אומן על Close ולא על Adj Close
    if "Adj_Close" in data.columns:
        data = data.drop(columns=["Adj_Close"])

    data["Symbol"] = symbol

    required_columns = [
        "Date",
        "Symbol",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Yahoo data is missing columns: {missing_columns}"
        )

    return data

In [18]:
aapl_raw = download_live_stock("AAPL")

print("Shape:", aapl_raw.shape)
print("Columns:", aapl_raw.columns.tolist())
display(aapl_raw.tail())

Shape: (501, 9)
Columns: ['Date', 'Close', 'Dividends', 'High', 'Low', 'Open', 'Stock_Splits', 'Volume', 'Symbol']


,Date,Close,Dividends,High,Low,Open,Stock_Splits,Volume,Symbol
496,2026-07-06,312.660004,0.0,314.200012,307.000000,307.359985,0.0,53590000,AAPL
497,2026-07-07,310.660004,0.0,315.480011,310.149994,315.290009,0.0,42490000,AAPL
498,2026-07-08,313.390015,0.0,314.820007,307.049988,311.910004,0.0,41323500,AAPL
499,2026-07-09,316.220001,0.0,316.529999,308.160004,310.510010,0.0,48124500,AAPL
500,2026-07-10,315.320007,0.0,316.910004,312.170013,314.720001,0.0,34109200,AAPL


In [19]:
import json
import joblib

MODELS_FOLDER = (
    "/content/drive/MyDrive/SWINGPULSE/models"
)

MODEL_PATH = (
    f"{MODELS_FOLDER}/random_forest_model.joblib"
)

METADATA_PATH = (
    f"{MODELS_FOLDER}/model_metadata.json"
)

model = joblib.load(MODEL_PATH)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:
    metadata = json.load(file)

FEATURE_COLUMNS = metadata["features"]
DECISION_THRESHOLD = metadata["decision_threshold"]
PREDICTION_HORIZON = metadata["prediction_horizon"]
TARGET_RETURN = metadata["target_return"]

print("Model loaded successfully")
print("Selected model:", metadata["selected_model"])
print("Features:", len(FEATURE_COLUMNS))
print("Decision threshold:", DECISION_THRESHOLD)
print("Target return:", TARGET_RETURN)
print("Prediction horizon:", PREDICTION_HORIZON)

Model loaded successfully
Selected model: Random Forest
Features: 19
Decision threshold: 0.45
Target return: 0.1
Prediction horizon: 10


In [20]:
import numpy as np


def analyze_live_stock(symbol: str) -> dict:
    symbol = str(symbol).upper().strip()

    try:
        live_data = download_live_stock(
            symbol=symbol,
            period="2y"
        )

        live_features = add_technical_features(
            live_data
        )

        valid_rows = live_features.dropna(
            subset=FEATURE_COLUMNS
        )

        if valid_rows.empty:
            return {
                "success": False,
                "message": (
                    f"Not enough historical data to calculate "
                    f"all model features for {symbol}."
                )
            }

        latest_row = valid_rows.iloc[-1]

        model_input = pd.DataFrame(
            [
                latest_row[
                    FEATURE_COLUMNS
                ].to_dict()
            ],
            columns=FEATURE_COLUMNS
        )

        model_input = model_input.replace(
            [np.inf, -np.inf],
            np.nan
        )

        if model_input.isna().any().any():
            return {
                "success": False,
                "message": (
                    f"The latest feature row for {symbol} "
                    "contains invalid values."
                )
            }

        probability = model.predict_proba(
            model_input
        )[0, 1]

        prediction = int(
            probability >= DECISION_THRESHOLD
        )

        if probability >= DECISION_THRESHOLD + 0.15:
            signal = "Strong Watch"
        elif probability >= DECISION_THRESHOLD:
            signal = "Watch"
        elif probability >= DECISION_THRESHOLD - 0.10:
            signal = "Neutral"
        else:
            signal = "Low Potential"

        return {
            "success": True,
            "symbol": symbol,
            "date": latest_row["Date"],
            "close": float(latest_row["Close"]),
            "probability": float(probability),
            "prediction": prediction,
            "signal": signal,
            "rsi": float(latest_row["RSI_14"]),
            "macd": float(latest_row["MACD"]),
            "macd_signal": float(
                latest_row["MACD_Signal"]
            ),
            "ema_20": float(latest_row["EMA_20"]),
            "ema_50": float(latest_row["EMA_50"]),
            "ema_200": float(latest_row["EMA_200"]),
            "atr": float(latest_row["ATR_14"]),
            "volatility": float(
                latest_row["Volatility_20"]
            )
        }

    except Exception as error:
        return {
            "success": False,
            "message": str(error)
        }

In [21]:
def display_live_analysis(symbol: str):
    result = analyze_live_stock(symbol)

    if not result["success"]:
        print(result["message"])
        return

    if result["rsi"] >= 70:
        rsi_status = "Overbought"
    elif result["rsi"] <= 30:
        rsi_status = "Oversold"
    elif result["rsi"] >= 55:
        rsi_status = "Positive momentum"
    elif result["rsi"] <= 45:
        rsi_status = "Weak momentum"
    else:
        rsi_status = "Neutral momentum"

    macd_status = (
        "Bullish"
        if result["macd"] > result["macd_signal"]
        else "Bearish"
    )

    if (
        result["ema_20"]
        > result["ema_50"]
        > result["ema_200"]
    ):
        trend = "Strong bullish trend"

    elif (
        result["ema_20"]
        < result["ema_50"]
        < result["ema_200"]
    ):
        trend = "Strong bearish trend"

    elif result["ema_20"] > result["ema_50"]:
        trend = "Short-term bullish trend"

    else:
        trend = "Mixed trend"

    print("=" * 64)
    print("SWINGPULSE LIVE STOCK ANALYSIS")
    print("=" * 64)

    print(f"Symbol:              {result['symbol']}")
    print(f"Yahoo data date:     {result['date']}")
    print(f"Latest close:        ${result['close']:.2f}")

    print("\nMODEL PREDICTION")
    print("-" * 64)

    print(
        f"Probability of +{TARGET_RETURN * 100:.0f}% "
        f"within {PREDICTION_HORIZON} trading days: "
        f"{result['probability'] * 100:.2f}%"
    )

    print(
        f"Decision threshold:  "
        f"{DECISION_THRESHOLD * 100:.0f}%"
    )

    print(f"Signal:              {result['signal']}")

    print("\nTECHNICAL INDICATORS")
    print("-" * 64)

    print(
        f"RSI (14):            "
        f"{result['rsi']:.2f} ({rsi_status})"
    )

    print(f"MACD:                {result['macd']:.4f}")

    print(
        f"MACD Signal:         "
        f"{result['macd_signal']:.4f}"
    )

    print(f"MACD status:         {macd_status}")
    print(f"Trend:               {trend}")
    print(f"ATR (14):            {result['atr']:.4f}")

    print(
        f"Volatility (20):     "
        f"{result['volatility']:.2%}"
    )

    print("\nDISCLAIMER")
    print("-" * 64)
    print(
        "This result is generated from historical market data "
        "and an experimental machine-learning model. "
        "It is not financial advice."
    )

In [22]:
display_live_analysis("AAPL")

SWINGPULSE LIVE STOCK ANALYSIS
Symbol:              AAPL
Yahoo data date:     2026-07-10 00:00:00
Latest close:        $315.32

MODEL PREDICTION
----------------------------------------------------------------
Probability of +10% within 10 trading days: 27.03%
Decision threshold:  45%
Signal:              Low Potential

TECHNICAL INDICATORS
----------------------------------------------------------------
RSI (14):            62.93 (Positive momentum)
MACD:                4.6148
MACD Signal:         1.8940
MACD status:         Bullish
Trend:               Strong bullish trend
ATR (14):            8.9464
Volatility (20):     2.21%

DISCLAIMER
----------------------------------------------------------------
This result is generated from historical market data and an experimental machine-learning model. It is not financial advice.


In [23]:
display_live_analysis("NVDA")

SWINGPULSE LIVE STOCK ANALYSIS
Symbol:              NVDA
Yahoo data date:     2026-07-10 00:00:00
Latest close:        $210.96

MODEL PREDICTION
----------------------------------------------------------------
Probability of +10% within 10 trading days: 45.98%
Decision threshold:  45%
Signal:              Watch

TECHNICAL INDICATORS
----------------------------------------------------------------
RSI (14):            56.97 (Positive momentum)
MACD:                -1.7351
MACD Signal:         -2.9782
MACD status:         Bullish
Trend:               Mixed trend
ATR (14):            6.8021
Volatility (20):     2.27%

DISCLAIMER
----------------------------------------------------------------
This result is generated from historical market data and an experimental machine-learning model. It is not financial advice.


In [24]:
display_live_analysis("TSLA")

SWINGPULSE LIVE STOCK ANALYSIS
Symbol:              TSLA
Yahoo data date:     2026-07-10 00:00:00
Latest close:        $407.76

MODEL PREDICTION
----------------------------------------------------------------
Probability of +10% within 10 trading days: 51.77%
Decision threshold:  45%
Signal:              Watch

TECHNICAL INDICATORS
----------------------------------------------------------------
RSI (14):            51.22 (Neutral momentum)
MACD:                -0.1438
MACD Signal:         -1.3355
MACD status:         Bullish
Trend:               Mixed trend
ATR (14):            20.1736
Volatility (20):     3.83%

DISCLAIMER
----------------------------------------------------------------
This result is generated from historical market data and an experimental machine-learning model. It is not financial advice.


In [25]:
display_live_analysis("NOTREAL")

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTREAL"}}}
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NOTREAL']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')


No Yahoo Finance data returned for NOTREAL.


In [26]:
LATEST_FEATURES_PATH = (
    "/content/drive/MyDrive/SWINGPULSE/"
    "data/features/SWINGPULSE_LATEST_FEATURES.csv"
)

symbol_source = pd.read_csv(
    LATEST_FEATURES_PATH,
    usecols=["Symbol"]
)

MARKET_SYMBOLS = (
    symbol_source["Symbol"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Number of symbols:", len(MARKET_SYMBOLS))
print(MARKET_SYMBOLS[:20])

Number of symbols: 501
['A', 'AAL', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AFL', 'AIG', 'AIZ', 'AJG']


In [27]:
import time


def scan_live_rsi(
    symbols,
    minimum_rsi=26,
    maximum_rsi=35,
    pause_seconds=0.4,
    maximum_results=None
):
    """
    סורקת מניות באמצעות Yahoo Finance ומחזירה מניות
    שה-RSI האחרון שלהן נמצא בטווח המבוקש.
    """

    results = []
    failed_symbols = []

    total_symbols = len(symbols)

    for index, symbol in enumerate(symbols, start=1):
        try:
            live_data = download_live_stock(
                symbol=symbol,
                period="1y"
            )

            live_features = add_technical_features(
                live_data
            )

            valid_rows = live_features.dropna(
                subset=[
                    "RSI_14",
                    "Close",
                    "MACD",
                    "MACD_Signal",
                    "EMA_20",
                    "EMA_50",
                    "Volatility_20"
                ]
            )

            if valid_rows.empty:
                failed_symbols.append(symbol)
                continue

            latest_row = valid_rows.iloc[-1]
            latest_rsi = float(latest_row["RSI_14"])

            if minimum_rsi <= latest_rsi <= maximum_rsi:
                macd_status = (
                    "Bullish"
                    if latest_row["MACD"]
                    > latest_row["MACD_Signal"]
                    else "Bearish"
                )

                trend_status = (
                    "Bullish"
                    if latest_row["EMA_20"]
                    > latest_row["EMA_50"]
                    else "Bearish"
                )

                results.append({
                    "Symbol": symbol,
                    "Date": latest_row["Date"],
                    "Close": float(latest_row["Close"]),
                    "RSI_14": latest_rsi,
                    "MACD_Status": macd_status,
                    "Trend": trend_status,
                    "Volatility_20": float(
                        latest_row["Volatility_20"]
                    )
                })

                if (
                    maximum_results is not None
                    and len(results) >= maximum_results
                ):
                    break

        except Exception as error:
            failed_symbols.append(symbol)

        if index % 25 == 0:
            print(
                f"Scanned {index}/{total_symbols} stocks | "
                f"Matches: {len(results)}"
            )

        time.sleep(pause_seconds)

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df = (
            results_df
            .sort_values(
                by="RSI_14",
                key=lambda values: (
                    values - 30
                ).abs()
            )
            .reset_index(drop=True)
        )

    print("\nScan completed")
    print("Matches found:", len(results_df))
    print("Failed symbols:", len(failed_symbols))

    return results_df, failed_symbols

In [28]:
rsi_results, failed_symbols = scan_live_rsi(
    symbols=MARKET_SYMBOLS,
    minimum_rsi=26,
    maximum_rsi=35
)

Scanned 25/501 stocks | Matches: 1


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ANSS']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ATVI']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data f

Scanned 50/501 stocks | Matches: 1
Scanned 75/501 stocks | Matches: 1


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CDAY']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')


Scanned 100/501 stocks | Matches: 1


...

Scanned 400/501 stocks | Matches: 4


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SEE']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')


Scanned 425/501 stocks | Matches: 4
Scanned 450/501 stocks | Matches: 4
Scanned 475/501 stocks | Matches: 5


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WRK']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')


Scanned 500/501 stocks | Matches: 5

Scan completed
Matches found: 5
Failed symbols: 23


In [29]:
if rsi_results.empty:
    print("No stocks were found in the requested RSI range.")

else:
    display(
        rsi_results.style.format({
            "Close": "${:.2f}",
            "RSI_14": "{:.2f}",
            "Volatility_20": "{:.2%}"
        })
    )

,Symbol,Date,Close,RSI_14,MACD_Status,Trend,Volatility_20
0,EQT,2026-07-10 00:00:00,$48.85,30.38,Bearish,Bearish,1.58%
1,ORCL,2026-07-10 00:00:00,$140.64,30.86,Bearish,Bearish,3.03%
2,ALB,2026-07-10 00:00:00,$126.05,27.88,Bearish,Bearish,3.57%
3,VZ,2026-07-10 00:00:00,$42.12,34.32,Bearish,Bearish,2.00%
4,CPRT,2026-07-10 00:00:00,$27.52,34.58,Bearish,Bearish,2.77%


In [30]:
def add_model_predictions_to_scan(scan_results):
    if scan_results.empty:
        return scan_results.copy()

    enriched_results = []

    for _, scan_row in scan_results.iterrows():
        symbol = scan_row["Symbol"]

        live_result = analyze_live_stock(symbol)

        if not live_result["success"]:
            continue

        enriched_results.append({
            "Symbol": symbol,
            "Date": live_result["date"],
            "Close": live_result["close"],
            "RSI_14": live_result["rsi"],
            "Probability": live_result["probability"],
            "Signal": live_result["signal"],
            "MACD": live_result["macd"],
            "MACD_Signal": live_result["macd_signal"],
            "Volatility_20": live_result["volatility"]
        })

    result_df = pd.DataFrame(enriched_results)

    if result_df.empty:
        return result_df

    result_df["Probability_Percent"] = (
        result_df["Probability"]
        .mul(100)
        .round(2)
    )

    return (
        result_df
        .sort_values(
            "Probability",
            ascending=False
        )
        .reset_index(drop=True)
    )

In [31]:
ranked_rsi_results = add_model_predictions_to_scan(
    rsi_results
)

display(
    ranked_rsi_results[
        [
            "Symbol",
            "Date",
            "Close",
            "RSI_14",
            "Probability_Percent",
            "Signal",
            "MACD",
            "Volatility_20"
        ]
    ].style.format({
        "Close": "${:.2f}",
        "RSI_14": "{:.2f}",
        "Probability_Percent": "{:.2f}%",
        "MACD": "{:.4f}",
        "Volatility_20": "{:.2%}"
    })
)

,Symbol,Date,Close,RSI_14,Probability_Percent,Signal,MACD,Volatility_20
0,CPRT,2026-07-10 00:00:00,$27.52,34.58,56.63%,Watch,-0.8893,2.77%
1,ALB,2026-07-10 00:00:00,$126.05,27.88,51.29%,Watch,-10.9523,3.57%
2,ORCL,2026-07-10 00:00:00,$140.64,30.86,47.72%,Watch,-14.1968,3.03%
3,VZ,2026-07-10 00:00:00,$42.12,34.32,32.07%,Low Potential,-1.2739,2.00%
4,EQT,2026-07-10 00:00:00,$48.85,30.38,26.64%,Low Potential,-1.0519,1.58%


In [32]:
def ask_rsi_agent(
    minimum_rsi=26,
    maximum_rsi=35
):
    print(
        f"Scanning stocks with RSI between "
        f"{minimum_rsi} and {maximum_rsi}..."
    )

    scan_results, failures = scan_live_rsi(
        symbols=MARKET_SYMBOLS,
        minimum_rsi=minimum_rsi,
        maximum_rsi=maximum_rsi
    )

    ranked_results = add_model_predictions_to_scan(
        scan_results
    )

    if ranked_results.empty:
        print("No matching stocks were found.")
        return

    display(
        ranked_results[
            [
                "Symbol",
                "Date",
                "Close",
                "RSI_14",
                "Probability_Percent",
                "Signal"
            ]
        ].style.format({
            "Close": "${:.2f}",
            "RSI_14": "{:.2f}",
            "Probability_Percent": "{:.2f}%"
        })
    )

In [33]:
ask_rsi_agent(26, 35)

Scanning stocks with RSI between 26 and 35...
Scanned 25/501 stocks | Matches: 1


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ANSS']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ATVI']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')


Scanned 50/501 stocks | Matches: 1
Scanned 75/501 stocks | Matches: 1


KeyboardInterrupt: 

In [34]:
import logging

logging.getLogger("yfinance").setLevel(logging.CRITICAL)

In [35]:
ask_rsi_agent(
    minimum_rsi=26,
    maximum_rsi=35
)

Scanning stocks with RSI between 26 and 35...
Scanned 25/501 stocks | Matches: 1
Scanned 50/501 stocks | Matches: 1
Scanned 75/501 stocks | Matches: 1
Scanned 100/501 stocks | Matches: 1
Scanned 125/501 stocks | Matches: 2
Scanned 150/501 stocks | Matches: 2
Scanned 175/501 stocks | Matches: 3
Scanned 200/501 stocks | Matches: 3
Scanned 225/501 stocks | Matches: 3
Scanned 250/501 stocks | Matches: 3
Scanned 275/501 stocks | Matches: 3
Scanned 300/501 stocks | Matches: 3
Scanned 325/501 stocks | Matches: 3
Scanned 350/501 stocks | Matches: 3
Scanned 375/501 stocks | Matches: 4
Scanned 400/501 stocks | Matches: 4
Scanned 425/501 stocks | Matches: 4
Scanned 450/501 stocks | Matches: 4
Scanned 475/501 stocks | Matches: 5
Scanned 500/501 stocks | Matches: 5

Scan completed
Matches found: 5
Failed symbols: 23


,Symbol,Date,Close,RSI_14,Probability_Percent,Signal
0,CPRT,2026-07-10 00:00:00,$27.52,34.58,56.63%,Watch
1,ALB,2026-07-10 00:00:00,$126.05,27.88,51.29%,Watch
2,ORCL,2026-07-10 00:00:00,$140.64,30.86,47.72%,Watch
3,VZ,2026-07-10 00:00:00,$42.12,34.32,32.07%,Low Potential
4,EQT,2026-07-10 00:00:00,$48.85,30.38,26.64%,Low Potential
